# Nollywood Movie Recommender
**3MTT NextGen AI/ML Project**

**Problem:** Viewers struggle to find Nigerian films they will enjoy.

**Approach:** Content-based filtering using TF-IDF (text similarity) on each movie's genre and lead actor, then ranking recommendations with cosine similarity. This is a standard, real recommender technique — no deep learning needed for an MVP of this scope.

**Steps in this notebook:**
1. Load the dataset
2. Build the TF-IDF feature model
3. Recommend function
4. Try it out
5. Evaluate the model


In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- Load your dataset ---
# Upload movies.csv to Colab (left sidebar > Files > upload),
# then point this at it. Required columns: title, genre, lead_actor, year
df = pd.read_csv("movies.csv")
df.head()

,title,genre,lead_actor,year,tag
0,Living in Bondage,"Drama, Thriller","Kenneth Okonkwo, Kanayo O. Kanayo",2026,supernatural thriller
1,Circle of Doom,Drama,"Richard Mofe-Damijo, Kanayo O. Kanayo",1993,crime drama
2,Glamour Girls,Drama,"Eucharia Anunobi, Liz Benson",1994,cult classic drama
3,Nneka the Pretty Serpent,"Horror, Drama","Ndidi Obi, Okechukwu Ogunjiofor",1994,supernatural horror
4,Rattlesnake,"Action, Crime","Francis Duru, Nkem Owoh",1995,action crime story


## 2. Build the TF-IDF model

TF-IDF (Term Frequency–Inverse Document Frequency) turns text into numbers so we can mathematically compare movies. We combine each movie's genre and lead actor into one text string (a "feature soup"), then let TF-IDF learn which words matter most for telling movies apart.

Cosine similarity then measures how close two movies are in that numeric space — 1.0 means identical, 0 means nothing in common.

In [8]:
df["features"] = df["genre"].astype(str) + ", " + df["lead_actor"].astype(str)

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df["features"])

# This is the "trained model": a matrix of how similar every movie is to every other movie
sim_matrix = cosine_similarity(tfidf_matrix)
print("Similarity matrix shape:", sim_matrix.shape)

Similarity matrix shape: (100, 100)


## 3. Recommend function

Given a movie title, this looks up its row in the similarity matrix, sorts every other movie by similarity score, and returns the top N matches.

In [9]:
def recommend(title, top_n=5):
    matches = df.index[df["title"].str.lower() == title.lower()]
    if len(matches) == 0:
        return f"'{title}' not found in dataset."
    idx = matches[0]

    scores = list(enumerate(sim_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [s for s in scores if s[0] != idx][:top_n]

    result = df.iloc[[i for i, _ in scores]][["title", "genre", "lead_actor", "year"]].copy()
    result["similarity_score"] = [round(score, 3) for _, score in scores]
    return result.reset_index(drop=True)

## 4. Try it out

Swap the title below for any movie in your dataset.

In [10]:
recommend("Sugar Rush")

,title,genre,lead_actor,year,similarity_score
0,Muri and Ko,"Comedy, Action","Kunle Remi, Bisola Aiyeola",2024,0.585
1,Gingerrr,Comedy,"Lateef Adedimeji, Bisola Aiyeola",2026,0.566
2,Gangs of Lagos,"Action, Crime","Tobi Bakre, Adesua Etomi",2023,0.464
3,The Wedding Party,"Comedy, Romance","Adesua Etomi, Banky Wellington",2016,0.435
4,The King Boys,"Crime, Drama","Sola Sobowale, Adesua Etomi",2018,0.401


## 5. Evaluate the model

The brief asks for evaluation results. Since this is unsupervised (no "correct answer" labels), we evaluate with two simple, honest checks instead of accuracy/precision:

1. **Coverage** — what fraction of movies get at least one recommendation with a similarity score above 0?
2. **Genre consistency** — for a sample of movies, what fraction of their top recommendation shares at least one genre with the original? (A sanity check that the model isn't recommending randomly.)

In [11]:
# --- Coverage ---
covered = 0
for i in range(len(df)):
    scores = [s for j, s in enumerate(sim_matrix[i]) if j != i]
    if max(scores) > 0:
        covered += 1
coverage = covered / len(df)
print(f"Coverage: {coverage:.2%} of movies have at least one meaningful recommendation")

# --- Genre consistency (sanity check) ---
matches = 0
for title in df["title"]:
    recs = recommend(title, top_n=1)
    if isinstance(recs, str):
        continue
    original_genres = set(g.strip() for g in df.loc[df["title"] == title, "genre"].values[0].split(","))
    rec_genres = set(g.strip() for g in recs.iloc[0]["genre"].split(","))
    if original_genres & rec_genres:
        matches += 1
consistency = matches / len(df)
print(f"Genre consistency: {consistency:.2%} of top recommendations share a genre with the original")

Coverage: 100.00% of movies have at least one meaningful recommendation
Genre consistency: 75.00% of top recommendations share a genre with the original


## Conclusion

This notebook builds a working content-based recommender for Nollywood movies using TF-IDF and cosine similarity. Given any movie title in the dataset, it returns the top 5 most similar titles based on shared genre and lead actor, along with a similarity score.

**Next step:** replace the sample data load with your compiled `movies.csv`, re-run all cells, and record the demo video walking through this notebook.